In [1]:
import torch
import torch.nn as nn
from pathlib import Path
import sys
from torchvision.transforms import transforms
from functools import partial
from torch.utils.data import DataLoader

In [2]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

PosixPath('/home/chiendoan/Code_Module6/Object_Detection_Part2')

In [3]:
sys.path.append(str(ROOT_DIR))
sys.path

['/home/chiendoan/miniconda3/envs/study_env/lib/python314.zip',
 '/home/chiendoan/miniconda3/envs/study_env/lib/python3.14',
 '/home/chiendoan/miniconda3/envs/study_env/lib/python3.14/lib-dynload',
 '',
 '/home/chiendoan/miniconda3/envs/study_env/lib/python3.14/site-packages',
 '/home/chiendoan/Code_Module6/Object_Detection_Part2']

In [4]:
DATA_DIR = str(ROOT_DIR/'data')

In [5]:
from model.FasterRCNN import *

/home/chiendoan/miniconda3/envs/study_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from src.orgainize_data import *

In [7]:
main_dir = Path(ROOT_DIR/'data'/'VOCdevkit'/'VOC2007'/'ImageSets'/'Main')
# Using set to collect single value
CLASSES = sorted({
    f.rsplit("_", 1)[0]
    for f in os.listdir(main_dir)
    if "_" in f and f.endswith(".txt")
})
CLASSES

['aeroplane',
 'bicycle',
 'bird',
 'boat',
 'bottle',
 'bus',
 'car',
 'cat',
 'chair',
 'cow',
 'diningtable',
 'dog',
 'horse',
 'motorbike',
 'person',
 'pottedplant',
 'sheep',
 'sofa',
 'train',
 'tvmonitor']

In [8]:
CLASSES2IDX = {clss: (idx+1) for idx, clss in enumerate(CLASSES)}
CLASSES2IDX

{'aeroplane': 1,
 'bicycle': 2,
 'bird': 3,
 'boat': 4,
 'bottle': 5,
 'bus': 6,
 'car': 7,
 'cat': 8,
 'chair': 9,
 'cow': 10,
 'diningtable': 11,
 'dog': 12,
 'horse': 13,
 'motorbike': 14,
 'person': 15,
 'pottedplant': 16,
 'sheep': 17,
 'sofa': 18,
 'train': 19,
 'tvmonitor': 20}

In [9]:
dataset = VOCObjectDetectionDataset(
    data_dir=DATA_DIR,
    split='trainval',
    list_transforms=[transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )],
    label2idx=CLASSES2IDX
)

In [10]:
dataset[0]

(tensor([[[-1.9627e+00, -1.9371e+00, -1.9296e+00,  ...,  6.9484e-01,
            6.7025e-01,  6.5096e-01],
          [-1.5904e+00, -1.8538e+00, -1.9490e+00,  ...,  6.9158e-01,
            6.9912e-01,  6.9912e-01],
          [-2.9198e-01, -3.4027e-01, -5.2594e-01,  ...,  7.2685e-01,
            7.2594e-01,  6.8736e-01],
          ...,
          [-1.8300e+00, -1.8172e+00, -1.6789e+00,  ..., -1.0259e+00,
           -1.1064e+00, -1.1771e+00],
          [-1.5517e+00, -1.5647e+00, -1.4882e+00,  ..., -1.1397e+00,
           -1.0766e+00, -1.0636e+00],
          [-1.3012e+00, -1.0632e+00, -7.1163e-01,  ..., -1.1077e+00,
           -9.9639e-01, -9.3524e-01]],
 
         [[-1.8715e+00, -1.8487e+00, -1.8365e+00,  ...,  1.2490e+00,
            1.2523e+00,  1.2326e+00],
          [-1.4385e+00, -1.7406e+00, -1.8630e+00,  ...,  1.2819e+00,
            1.2819e+00,  1.2819e+00],
          [ 2.1332e-02, -4.7703e-02, -2.6379e-01,  ...,  1.3158e+00,
            1.3005e+00,  1.2611e+00],
          ...,
    

In [11]:
collate = partial(collate_fn)

In [12]:
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate
)

In [13]:
Detector = FasterRCNN_model(num_classes=len(CLASSES))

In [15]:
from statistics import mode
trainer = TrainModel(
    model=Detector,
    dataloader=dataloader,
    n_epochs=10,
    batch_size=16
)

In [18]:
PATH_SAVE_WEIGHT = ROOT_DIR/'model_weight.pth'
torch.save(Detector.state_dict(), Path(PATH_SAVE_WEIGHT))

In [17]:
trainer()

Training:  10%|█         | 1/10 [13:22<2:00:26, 802.98s/it]

Epoch [1/10] - Total Loss: 0.8143 | RPN (cls: 0.0149, reg: 0.2175, acc: 99.81%) | Fast R-CNN (cls: 0.4151, reg: 0.1667, acc: 88.94%)


Training:  20%|██        | 2/10 [26:37<1:46:26, 798.29s/it]

Epoch [2/10] - Total Loss: 0.5053 | RPN (cls: 0.0078, reg: 0.1343, acc: 99.88%) | Fast R-CNN (cls: 0.3151, reg: 0.0482, acc: 89.70%)


Training:  30%|███       | 3/10 [39:52<1:32:54, 796.40s/it]

...

Epoch [8/10] - Total Loss: 0.3244 | RPN (cls: 0.0059, reg: 0.0663, acc: 99.88%) | Fast R-CNN (cls: 0.2137, reg: 0.0386, acc: 92.27%)


Training:  90%|█████████ | 9/10 [1:57:57<12:54, 774.06s/it]

Epoch [9/10] - Total Loss: 0.3089 | RPN (cls: 0.0058, reg: 0.0622, acc: 99.88%) | Fast R-CNN (cls: 0.2034, reg: 0.0375, acc: 92.56%)


Training: 100%|██████████| 10/10 [2:10:38<00:00, 783.84s/it]

Epoch [10/10] - Total Loss: 0.2939 | RPN (cls: 0.0057, reg: 0.0580, acc: 99.88%) | Fast R-CNN (cls: 0.1935, reg: 0.0367, acc: 92.84%)
